# CNN output-shape calculator and forward-hook drill

This notebook is the template for Practice Ladder row 11. Complete every `TODO` without looking up the resulting shapes. Predict them on paper first, then use PyTorch to verify them.

## Completion target

1. Implement the output-size formula.
2. Predict every layer's `(N, C, H, W)` shape.
3. Add forward hooks that record the real output shapes.
4. Calculate the flattened feature count.
5. Explain any prediction that did not match.

In [ ]:
import math
import torch
from torch import nn

torch.manual_seed(42)
print('PyTorch:', torch.__version__)

PyTorch: 2.11.0+cpu


## 1. Write the formula

For one spatial dimension, PyTorch uses:

$$
L_{out}=\left\lfloor\frac{L_{in}+2p-d(k-1)-1}{s}+1\right\rfloor
$$

where `k` is kernel size, `s` is stride, `p` is padding, and `d` is dilation. Apply it separately to height and width. The batch size does not change. A convolution's output-channel count is its `out_channels`.

In [2]:
def output_size_1d(input_size, kernel_size, stride=1, padding=0, dilation=1):
    """Return the integer output length of a convolution or pooling operation."""
    # TODO: translate the formula above into Python.
    output_size = (input_size + 2*padding - dilation*(kernel_size-1) - 1)//stride + 1
    return output_size

In [3]:
def check_calculator():
    cases = [
        # (input, kernel, stride, padding, dilation)
        (10, 3, 1, 0, 1),
        (10, 3, 1, 1, 1),
        (15, 3, 2, 1, 1),
        (20, 3, 1, 2, 2),
    ]

    passed = 0
    for args in cases:
        result = output_size_1d(*args)
        # PyTorch is used only as the checker, not as the calculator.
        layer = nn.Conv1d(1, 1, kernel_size=args[1], stride=args[2],
                          padding=args[3], dilation=args[4])
        expected = layer(torch.zeros(1, 1, args[0])).shape[-1]
        ok = result == expected
        passed += int(ok)
        print(f'inputs={args}  your result={result}  status={"PASS" if ok else "RETRY"}')

    print(f'\nCalculator checks passed: {passed}/{len(cases)}')

check_calculator()

inputs=(10, 3, 1, 0, 1)  your result=8  status=PASS
inputs=(10, 3, 1, 1, 1)  your result=10  status=PASS
inputs=(15, 3, 2, 1, 1)  your result=8  status=PASS
inputs=(20, 3, 1, 2, 2)  your result=20  status=PASS

Calculator checks passed: 4/4


## 2. Predict the five layer shapes on paper

Input tensor: **`(4, 3, 64, 64)`**

| Name | Operation | Output shape prediction |
|---|---|---|
| `conv1` | `Conv2d(3, 8, kernel_size=3, stride=1, padding=1)` | write on paper |
| `pool1` | `MaxPool2d(kernel_size=2, stride=2)` | write on paper |
| `conv2` | `Conv2d(8, 16, kernel_size=5, stride=2, padding=2)` | write on paper |
| `pool2` | `MaxPool2d(kernel_size=3, stride=2, padding=1)` | write on paper |
| `conv3` | `Conv2d(16, 32, kernel_size=3, stride=1, padding=0)` | write on paper |

Include batch size and channels in every answer. Only run the following PyTorch verification after completing the paper table.

In [4]:
# TODO: replace each None with an integer from your paper calculations.
predicted_shapes = {
    'conv1': (4, 8, 64, 64),
    'pool1': (4, 8, 32, 32),
    'conv2': (4, 16, 16, 16),
    'pool2': (4, 16, 8, 8),
    'conv3': (4, 32, 6, 6),
}

predicted_shapes

{'conv1': (4, 8, 64, 64),
 'pool1': (4, 8, 32, 32),
 'conv2': (4, 16, 16, 16),
 'pool2': (4, 16, 8, 8),
 'conv3': (4, 32, 6, 6)}

## 3. Build the network

ReLU layers are included, but the hook exercise will record only convolution and pooling outputs. ReLU does not change tensor shape.

In [5]:
class ShapePracticeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(8, 16, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=0),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.features(x)

model = ShapePracticeCNN()
x = torch.randn(4, 3, 64, 64)
model

ShapePracticeCNN(
  (features): Sequential(
    (0): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(8, 16, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (6): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
    (7): ReLU()
  )
)

## 4. Add forward hooks

A forward hook runs automatically after a layer finishes its forward pass. It receives the layer, its inputs, and its output. Your hook should save `tuple(output.shape)` in `hook_log`.

Useful pattern: `layer.register_forward_hook(your_hook_function)`. Save every returned handle so you can remove the hooks afterward.

In [8]:
hook_log = {}

def make_shape_hook(name):
    def hook(module, inputs, output):
        # TODO: store this layer's output shape in hook_log under `name`.
        hook_log[name] = output.shape
    return hook

# Sequential positions of the five layers whose shapes we want.
layer_positions = {
    'conv1': 0,
    'pool1': 2,
    'conv2': 3,
    'pool2': 5,
    'conv3': 6,
}

handles = []
for name, position in layer_positions.items():
    layer = model.features[position]
    # TODO: register make_shape_hook(name) on `layer` and append the handle.
    handle = layer.register_forward_hook(make_shape_hook(name))
    handles.append(handle)

print('Number of registered hooks:', len(handles))

Number of registered hooks: 5


In [9]:
# Run one forward pass to trigger the hooks.
with torch.no_grad():
    final_feature_map = model(x)

print('Hook log:', hook_log)
print('Final feature-map shape:', tuple(final_feature_map.shape))

# Remove hooks so repeated runs do not accumulate them.
for handle in handles:
    handle.remove()

Hook log: {'conv1': torch.Size([4, 8, 64, 64]), 'pool1': torch.Size([4, 8, 32, 32]), 'conv2': torch.Size([4, 16, 16, 16]), 'pool2': torch.Size([4, 16, 8, 8]), 'conv3': torch.Size([4, 32, 6, 6])}
Final feature-map shape: (4, 32, 6, 6)


## 5. Compare your predictions with PyTorch

Do not edit your original predictions after seeing the real shapes. A mismatch is useful evidence: identify whether channels, padding, stride, or floor rounding caused it.

In [10]:
matches = 0

for name in predicted_shapes:
    predicted = predicted_shapes[name]
    actual = hook_log.get(name)
    is_match = predicted == actual
    matches += int(is_match)
    print(f'{name:6s} predicted={predicted} actual={actual}  ' +
          ('MATCH' if is_match else 'CHECK AGAIN'))

print(f'\nShape predictions correct: {matches}/{len(predicted_shapes)}')

conv1  predicted=(4, 8, 64, 64) actual=torch.Size([4, 8, 64, 64])  MATCH
pool1  predicted=(4, 8, 32, 32) actual=torch.Size([4, 8, 32, 32])  MATCH
conv2  predicted=(4, 16, 16, 16) actual=torch.Size([4, 16, 16, 16])  MATCH
pool2  predicted=(4, 16, 8, 8) actual=torch.Size([4, 16, 8, 8])  MATCH
conv3  predicted=(4, 32, 6, 6) actual=torch.Size([4, 32, 6, 6])  MATCH

Shape predictions correct: 5/5


## 6. Calculate the flattened feature count

For one image, the flattened feature count is `channels × height × width`. Do not include the batch dimension. Calculate it from your final predicted shape before running the check.

In [11]:
# TODO: replace None with your paper calculation.
flattened_features = 1152

actual_flattened_features = final_feature_map[0].numel()
print('Your flattened feature count:', flattened_features)
print('PyTorch feature count:       ', actual_flattened_features)
print('Status:', 'PASS' if flattened_features == actual_flattened_features else 'RETRY')

Your flattened feature count: 1152
PyTorch feature count:        1152
Status: PASS


## 7. Optional transfer check

After the main exercise passes, change only the input image size from `64 × 64` to `65 × 65`. Predict all five shapes again before running the model. This exposes where floor rounding matters.

In [12]:
# TODO (optional): fill these in on paper, then replace the None values.
predictions_for_65 = {
    'conv1': (4, None, None, None),
    'pool1': (4, None, None, None),
    'conv2': (4, None, None, None),
    'pool2': (4, None, None, None),
    'conv3': (4, None, None, None),
}

x_65 = torch.randn(4, 3, 65, 65)

# Reuse your hook-registration approach here if you want an automatic comparison.
with torch.no_grad():
    output_65 = model(x_65)

print('Final output for 65 x 65 input:', tuple(output_65.shape))

Final output for 65 x 65 input: (4, 32, 6, 6)


## Reflection

Write short answers before marking the exercise complete:

1. Which parameter determines the output-channel count of a convolution?
2. Why does ReLU not need a separate shape calculation?
3. At which layers did spatial resolution decrease?
4. Where did the floor operation matter?
5. Why must the batch dimension be excluded from `in_features` for a linear layer?